# RealPage Message Agent — LoRA Training (Colab Pro)

Fine-tune **Qwen2.5-1.5B-Instruct** on your exported chat JSONL.

**Before you start**
1. Runtime → **Change runtime type** → **T4 GPU** (or A100 if available on Pro+)
2. Run cells top to bottom
3. Upload `train.jsonl` in step 2 **or** place it on Google Drive first

Expected runtime: **~1–3 hours** (T4, 3000 rows, 2 epochs)

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Get training data

Pick **one** option below.

In [ ]:
from pathlib import Path

WORK = Path("/content/realpage-message-agent")
DATA = WORK / "data" / "finetune"
OUT = WORK / "models" / "realpage-message-agent-synthetic"
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA / "train.jsonl"

# --- Option A: upload from your PC (easiest) ---
from google.colab import files

print("Select train.jsonl from your PC (~6 MB)...")
uploaded = files.upload()
name = next(iter(uploaded))
TRAIN_FILE.write_bytes(uploaded[name])
print(f"Saved {TRAIN_FILE} ({TRAIN_FILE.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# --- Option B: read from Google Drive (skip Option A if you use this) ---
# Uncomment and set DRIVE_PATH to where you copied train.jsonl on Drive.

# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_PATH = "/content/drive/MyDrive/RealPage/data/finetune/train.jsonl"
# TRAIN_FILE.write_bytes(Path(DRIVE_PATH).read_bytes())
# print(f"Copied from Drive → {TRAIN_FILE}")

In [ ]:
# --- Option C: clone repo if pushed to GitHub ---
# !git clone https://github.com/YOUR_USER/realpage-message-agent.git /content/realpage-message-agent
# TRAIN_FILE = Path("/content/realpage-message-agent/data/finetune/train.jsonl")

## 2. Install dependencies

In [ ]:
!pip install -q "transformers>=4.46" "datasets>=3.0" "peft>=0.13" "trl>=0.12" "accelerate>=1.0" bitsandbytes sentencepiece protobuf

## 3. Training config

Adjust if needed. T4 16GB usually handles these defaults.

In [ ]:
CONFIG = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "epochs": 2,
    "max_length": 768,
    "batch_size": 2,
    "grad_accum": 4,
    "learning_rate": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_samples": None,  # set e.g. 500 for a quick test
    "merge": True,
    "smoke_test": True,
}
CONFIG

## 4. Train

In [ ]:
import json
import torch
from pathlib import Path
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]


def load_chat_dataset(path: Path) -> Dataset:
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    if not rows:
        raise ValueError(f"No rows in {path}")
    return Dataset.from_list(rows)


def save_merged_model(trainer, tokenizer, output_dir: Path, base_model: str) -> None:
    merged = trainer.model.merge_and_unload()
    merged.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    metadata = {
        "base_model": base_model,
        "format": "merged-lora",
        "task": "realpage-message-agent-json",
    }
    (output_dir / "realpage_model.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )


train_file = Path(TRAIN_FILE)
output_dir = Path(OUT)
output_dir.mkdir(parents=True, exist_ok=True)

dataset = load_chat_dataset(train_file)
if CONFIG["max_samples"]:
    dataset = dataset.select(range(min(CONFIG["max_samples"], len(dataset))))
print(f"Training examples: {len(dataset)}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

model_name = CONFIG["model"]
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

training_args = SFTConfig(
    output_dir=str(output_dir / "checkpoints"),
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["learning_rate"],
    logging_steps=10,
    save_strategy="no",
    max_length=CONFIG["max_length"],
    assistant_only_loss=True,
    bf16=False,
    fp16=False,
    gradient_checkpointing=True,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Starting LoRA training...")
result = trainer.train()
print(f"Done. Loss: {result.training_loss:.4f}")

adapter_dir = output_dir / "adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved: {adapter_dir}")

if CONFIG["merge"]:
    print("Merging adapter...")
    save_merged_model(trainer, tokenizer, output_dir, model_name)
    print(f"Merged model saved: {output_dir}")

## 5. Smoke test

In [ ]:
if CONFIG["smoke_test"]:
    sample = dataset[0]
    messages = sample["messages"][:2]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    trainer.model.eval()
    with torch.no_grad():
        out = trainer.model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    print(text[:2000])

## 7. Upload to Hugging Face Hub (recommended for Vercel)

Upload the **merged** model so you can deploy a GPU Inference Endpoint. Skip §6 PC download if you use this path.

Replace `YOUR_USERNAME` with your Hugging Face username.

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import HfApi, create_repo, notebook_login

notebook_login()  # paste a Write token from https://huggingface.co/settings/tokens

REPO_ID = "YOUR_USERNAME/realpage-message-agent-v1"  # <-- change this

api = HfApi()
create_repo(repo_id=REPO_ID, repo_type="model", private=True, exist_ok=True)

# Upload merged model folder (must contain .safetensors files)
api.upload_folder(
    folder_path=str(OUT),
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="RealPage merged LoRA model from Colab",
)

print(f"Uploaded: https://huggingface.co/{REPO_ID}")
print("Next: Hub → Deploy → Inference Endpoints (TGI). See docs/HOSTING.md")

## 6. Download model to your PC

Copy merged model **or** adapter folder back to:
`realpage-message-agent/models/realpage-message-agent-synthetic/`

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/realpage-message-agent-synthetic.zip"
shutil.make_archive(
    "/content/realpage-message-agent-synthetic",
    "zip",
    root_dir=str(OUT.parent),
    base_dir=OUT.name,
)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
files.download(zip_path)

In [ ]:
# Optional: also save to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/realpage-message-agent-synthetic.zip "/content/drive/MyDrive/RealPage/"